# 05 — Fine-tune YourMT3+ on the Restrudel corpus

Goal 2 of the project: **fine-tune the released YourMT3+ checkpoint** so it
transcribes synth-heavy electronic music, using the corpus assembled in
`04_finetune_data.ipynb`. This notebook covers:

1. **Corpus composition** — what data we will fine-tune on, by category (plots).
2. **Sampling mix** — how the target (Strudel) and forgetting-mitigation
   reference sets (EGMD / MAESTRO / Slakh) are weighted in a training batch.
3. **Register a `strudel` preset** in YourMT3's `amt/src/config/data_presets.py`.
4. **Baseline to beat** — run the released checkpoint on held-out real clips.
5. **Fine-tune** — short run on Colab (1 GPU, bf16, ≪300k steps) → Drive.
6. **Evaluate** — note-level F1 on the held-out *real* electronic set vs. baseline.

> Prereq: the datasets already live in Drive under `DATA_HOME` in YourMT3's
> 16 kHz load format (built by notebook 04). This notebook reads their index
> files; it does not re-download anything.

In [ ]:
# Environment: Colab (Drive-mounted) or local checkout.
import json, os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/restrudel")
    DATA_HOME = DRIVE / "datasets"
    if not Path("/content/restrudel").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/henrik253/Restrudel.git", "/content/restrudel"], check=True)
    REPO = Path("/content/restrudel")
else:
    REPO = Path(os.environ.get("RESTRUDEL_REPO", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
    DATA_HOME = Path(os.environ.get("RESTRUDEL_DATA_HOME", REPO / "datasets"))

INDEX_DIR = DATA_HOME / "yourmt3_indexes"
AMT_SRC = Path(os.environ.get("RESTRUDEL_AMT_SRC", REPO / "models" / "YourMT3" / "amt" / "src"))
print(f"colab={IN_COLAB}\nrepo={REPO}\ndata_home={DATA_HOME}\nindexes={INDEX_DIR}")
assert INDEX_DIR.exists(), "no index dir — run notebook 04 first to build the datasets in Drive"

## 1. Fine-tuning corpus — composition by category

The corpus has two roles:

- **Target domain** — `strudel`: real corpus snippets + LLM-enhanced synthetic
  songs. Synth/electronic timbres — the thing the stock model fails on.
- **Forgetting-mitigation (reference)** — `egmd`, `maestro`, `slakh`: keep the
  model's general transcription ability while we specialise it.

The cell below reads every `*_file_list.json` and aggregates songs, audio hours
(`n_frames / 16000 / 3600`) and on-disk GB per dataset.

In [ ]:
# Aggregate the corpus from the index files (train+val+test only).
import subprocess

META = {  # dataset -> (label, role, domain, color)
    "strudel": ("Strudel (synth/electronic)", "target",    "electronic", "#d1495b"),
    "egmd":    ("EGMD (electronic drums)",     "reference", "electronic", "#edae49"),
    "maestro": ("MAESTRO (acoustic piano)",    "reference", "acoustic",   "#66a182"),
    "slakh":   ("Slakh (acoustic band)",       "reference", "acoustic",   "#2e4057"),
}
DIRNAME = {"slakh": "slakh2100_yourmt3_16k"}  # else f"{k}_yourmt3_16k"

def hours(fl):
    return sum(e.get("n_frames", 0) for e in fl.values()) / 16000 / 3600

agg = {k: {"train": 0, "validation": 0, "test": 0, "songs": 0, "hours": 0.0, "gb": 0.0} for k in META}
for p in sorted(INDEX_DIR.glob("*_file_list.json")):
    ds, _, split = p.stem.replace("_file_list", "").rpartition("_")
    if ds not in META or split not in ("train", "validation", "test"):
        continue
    fl = json.load(open(p))
    agg[ds][split] += len(fl)
    agg[ds]["songs"] += len(fl)
    agg[ds]["hours"] += hours(fl)
for k in META:
    d = DATA_HOME / DIRNAME.get(k, f"{k}_yourmt3_16k")
    out = subprocess.run(f"du -sb {d} 2>/dev/null", shell=True, capture_output=True, text=True).stdout
    agg[k]["gb"] = int(out.split()[0]) / 1e9 if out.strip() else 0.0

print(f"{'dataset':<10}{'role':<11}{'train':>7}{'val':>6}{'test':>6}{'songs':>8}{'hours':>8}{'GB':>7}")
print("-" * 63)
for k, (lab, role, dom, _) in META.items():
    a = agg[k]
    print(f"{k:<10}{role:<11}{a['train']:>7}{a['validation']:>6}{a['test']:>6}"
          f"{a['songs']:>8}{a['hours']:>8.1f}{a['gb']:>7.1f}")
tot = lambda m: sum(agg[k][m] for k in META)
print("-" * 63)
print(f"{'TOTAL':<21}{tot('train'):>7}{tot('validation'):>6}{tot('test'):>6}"
      f"{tot('songs'):>8}{tot('hours'):>8.1f}{tot('gb'):>7.1f}")

### 1a. Pie charts — size by dataset

Three views of the same corpus. Note how the metrics disagree: **Slakh** dominates
disk (multi-stem `.npy`), **EGMD** dominates song count and hours, while the
**target Strudel** data is tiny — which is exactly why the sampling mix (§2), not
the raw sizes, controls what the model actually sees.

In [ ]:
import matplotlib.pyplot as plt

order  = list(META)
labels = [META[k][0] for k in order]
colors = [META[k][3] for k in order]

def _autopct(vals):
    t = sum(vals)
    return lambda p: f"{p:.1f}%\n({p/100*t:,.0f})" if p > 4 else f"{p:.1f}%"

fig, axes = plt.subplots(1, 3, figsize=(16, 5.4))
for ax, (m, title) in zip(axes, [("songs", "# songs"), ("hours", "hours of audio"), ("gb", "disk size (GB)")]):
    vals = [agg[k][m] for k in order]
    ax.pie(vals, colors=colors, startangle=90, counterclock=False,
           autopct=_autopct(vals), pctdistance=0.72, textprops={"fontsize": 8},
           wedgeprops=dict(width=0.42, edgecolor="white"))
    ax.set_title(f"{title}  ·  total {sum(vals):,.0f}", fontsize=11)
fig.legend(labels, loc="lower center", ncol=4, frameon=False, fontsize=10)
fig.suptitle("Restrudel fine-tuning corpus — composition by dataset", fontsize=13, y=1.03)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

### 1b. Target vs. reference, and electronic vs. acoustic

The same hours regrouped two ways: by **training role** (what we are teaching vs.
what keeps the model from forgetting) and by **timbre domain**.

In [ ]:
from collections import defaultdict

def group(metric, keyfn):
    d = defaultdict(float)
    for k in META:
        d[keyfn(k)] += agg[k][metric]
    return d

by_role = group("hours", lambda k: "target (Strudel)" if META[k][1] == "target" else "reference (forgetting-mitigation)")
by_dom  = group("hours", lambda k: META[k][2])

fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(16, 5))
a1.pie(by_role.values(), labels=list(by_role), colors=["#d1495b", "#8d99ae"],
       autopct=lambda p: f"{p:.1f}%", startangle=90, counterclock=False,
       wedgeprops=dict(width=0.45, edgecolor="white"), textprops={"fontsize": 9})
a1.set_title("hours by role", fontsize=11)
a2.pie(by_dom.values(), labels=list(by_dom), colors=["#e07a5f", "#3d8361"],
       autopct=lambda p: f"{p:.1f}%", startangle=90, counterclock=False,
       wedgeprops=dict(width=0.45, edgecolor="white"), textprops={"fontsize": 9})
a2.set_title("hours by timbre domain", fontsize=11)

# stacked train/val/test hours per dataset (read each split's index directly)
import numpy as np
splits = ["train", "validation", "test"]
ph = {k: {s: 0.0 for s in splits} for k in order}
for k in order:
    for s in splits:
        fpath = INDEX_DIR / f"{k}_{s}_file_list.json"
        if fpath.exists():
            ph[k][s] = hours(json.load(open(fpath)))
bottom = np.zeros(len(order))
scol = {"train": "#2a9d8f", "validation": "#e9c46a", "test": "#e76f51"}
for s in splits:
    vals = [ph[k][s] for k in order]
    a3.bar([META[k][0].split(" (")[0] for k in order], vals, bottom=bottom, label=s, color=scol[s])
    bottom += vals
a3.set_ylabel("hours"); a3.set_title("train / val / test hours", fontsize=11)
a3.tick_params(axis="x", rotation=25, labelsize=8); a3.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 2. Sampling mix for fine-tuning

Fine-tuning **only** on Strudel would make the model forget everything else, so a
training batch mixes the target with the reference sets. YourMT3's original
`all_cross_final` preset weighted 10 datasets (see notebook 04 §4); here we need a
much simpler mix that **upweights the tiny target** relative to its raw size.

Below is a *starting* proposal — Strudel carries the bulk of the sampling weight
even though it is <1 % of the hours, with the reference sets sharing the rest to
hold general performance. Tune these before the real run.

In [ ]:
# Proposed sampling weights (sum to 1). NOT proportional to size — the target is upweighted.
WEIGHTS = {
    "strudel": 0.50,   # target domain — the point of the fine-tune
    "slakh":   0.25,   # richest multi-instrument reference
    "maestro": 0.15,   # pitched-note replay (piano)
    "egmd":    0.10,   # drum-kit coverage
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9

raw = {k: agg[k]["hours"] for k in order}
raw_share = {k: raw[k] / sum(raw.values()) for k in order}

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5))
a1.pie([raw_share[k] for k in order], labels=[k for k in order], colors=colors,
       autopct=lambda p: f"{p:.1f}%", startangle=90, counterclock=False,
       wedgeprops=dict(width=0.45, edgecolor="white"), textprops={"fontsize": 9})
a1.set_title("raw corpus share (by hours)", fontsize=11)
a2.pie([WEIGHTS[k] for k in order], labels=[k for k in order], colors=colors,
       autopct=lambda p: f"{p:.0f}%", startangle=90, counterclock=False,
       wedgeprops=dict(width=0.45, edgecolor="white"), textprops={"fontsize": 9})
a2.set_title("proposed sampling weight", fontsize=11)
fig.suptitle("Raw size vs. training sampling mix — the target is deliberately upweighted", y=1.02)
plt.tight_layout(); plt.show()

for k in order:
    print(f"{k:<9} raw {raw_share[k]*100:5.1f}%  ->  sampled {WEIGHTS[k]*100:4.0f}%")

## 3. Register a `strudel` preset in YourMT3  *(next step, stub)*

YourMT3's data loader reads named presets from `amt/src/config/data_presets.py`.
To fine-tune on our mix we add a `strudel` (target-only) preset and a
`strudel_ft` (target + reference, using §2 weights) preset pointing at the index
files in `INDEX_DIR`. Then the training entrypoint is called with that preset.

```python
# sketch — to be written into amt/src/config/data_presets.py
data_preset_single_cfg = {
  "strudel":   {"eval_vocab": [...], "train_split": "strudel_train",  "validation_split": "strudel_validation", "test_split": "strudel_test"},
}
data_preset_multi_cfg = {
  "strudel_ft": {"presets": ["strudel", "slakh", "maestro", "egmd"],
                 "weights": [0.50, 0.25, 0.15, 0.10]},   # from §2
}
```


## 4. Baseline to beat  *(next step, stub)*

Run the **released** `YPTF.MoE+Multi (noPS)` checkpoint on a handful of held-out
*real* electronic clips (`mp3s/` and the Strudel corpus **test** split) and record
note-level F1. This is the number the fine-tune must beat.

- checkpoint: `mimbres/YourMT3` space (536 MB) — not fetched in this notebook yet.
- inference entrypoint: `amt/src/` model runner; feed 16 kHz mono.

## 5. Fine-tune run  *(next step, stub)*

Short Colab run: 1 GPU, **bf16**, ≪300k steps, resume from the released checkpoint,
sampling from the `strudel_ft` preset, checkpoint back to Drive.

- mount Drive (done above), pull the index files (already in `INDEX_DIR`).
- confirm a GPU runtime (`torch.cuda.is_available()`), pin batch size to fit 16 GB.
- log train/val loss; snapshot to `DRIVE/checkpoints/`.

In [ ]:
# GPU sanity check for the eventual training run.
try:
    import torch
    print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
          "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU — set Runtime > Change runtime type > GPU"))
except ImportError:
    print("torch not installed in this runtime yet (only needed for §4–6)")

## 6. Evaluate vs. baseline  *(next step, stub)*

Note-level F1 (onset / onset+offset / multi-instrument) on the held-out **real**
electronic set — the same clips as §4 — for the fine-tuned checkpoint vs. the
baseline. A win here is the thesis result (Milestone M6).